In [41]:
import pandas as pd
import random
import itertools as it
import copy

# Enable to cross-check data: index value versus lat, lon : 
import folium
import json

## Data imports

In [ ]:
df = pd.read_excel('BRAILS_inventory.xlsx', header = 0, index_col = 0)
df

## Create archetype guesses

In [114]:
necessary_columns = ['PlanArea','NumberOfStories','StructureType','garageExists','roofshape','YearBuilt']
missing = [x for x in necessary_columns if x not in df.columns]


if len(missing) > 0 : 
    print('The following column headers are found missing and should be relabeled or new data is required :')
    print(missing)
    print('Current column labels :')
    print(list(df.columns))

The following column headers are found missing and should be relabeled or new data is required :
['PlanArea', 'NumberOfStories', 'StructureType', 'YearBuilt']
Current column labels :
['numstories', 'roofshape', 'footprintArea', 'chimneyExists', 'garageExists', 'Yearbuilt', 'Occupancy', 'ConstrType', 'Lat', 'Lon']


In [115]:
relabeler = {
    'footprintArea':'PlanArea',
    'numstories':'NumberOfStories',
    'ConstrType':'StructureType',
    'Yearbuilt':'YearBuilt',
}

In [ ]:
df = df.rename(columns=relabeler)
display(df)

In [118]:
URL = r'C:\PATH\TO\YOUR\LOCAL\GEOJSON\Geo_data_OS2.geojson'

with open(URL) as f:
    data = json.load(f)
    
for val in data['features']:
    val['properties']['id'] = val['id']

## Clean construction type / Structure type

In [119]:
# Available types : Wood (WOD), Steel (STL), Concrete (RCC), Masonry (MAS), Manufactured (MAB)

In [ ]:
center_lat = 44.65
center_lon = -63.55

# Initialize an interactive map centered on the footprints
m = folium.Map(
    location=(center_lat, center_lon),
    tiles="cartodbpositron",  # Light, clean basemap style
    zoom_start=13
)

folium.GeoJson(
    data,
    name="geojson",
    tooltip=folium.GeoJsonTooltip(fields=['id'], sticky=False)
).add_to(m)


# Using google maps on another browser page, check for MAB/MAS buildings to verify predictions
mask = (df['StructureType'] == 'MAB') | (df['StructureType']=='MAS')

mask = (df['StructureType'] == 'MAB')
#mask = (df['StructureType'] == 'MAS')
#mask = (df['StructureType'] == 'RCC')
#mask = (df['StructureType'] == 'STL')

df_mask = df.loc[mask,:]



df_mask.apply(lambda row: folium.CircleMarker(location=[row["Lat"], row["Lon"]], 
                                        radius=10,
                                        popup=[row['StructureType'],row.name],
                                       color = 'red'
                                       )
                                             .add_to(m), axis=1)

m

In [121]:
# Some identified records at not full sized buildings, while others are obvious missclassifications
# A clean up is made by inspecting the structure types manually (i.e. via google maps). 
clean_up = pd.read_excel('BRAILS - data cleanup.xlsx', index_col = 0, header = 0)
display(clean_up)

,StructureType,New Type
id,,
33,STL,WOD
140,STL,WOD
146,STL,WOD
8,RCC,WOD
25,RCC,WOD
...,...,...
59,MAB,WOD
153,-,DEL
152,-,DEL


In [ ]:
df.loc[clean_up.index,'StructureType'] = clean_up['New Type'].values
df = df[df['StructureType']!= 'DEL']
df

In [123]:
new_features = [val for val in data['features'] if int(val['id']) in list(df.index)]
filtered_data = data.copy()
filtered_data['features'] = new_features

In [ ]:
# Initialize an interactive map centered on the footprints
m = folium.Map(
    location=(center_lat, center_lon),
    tiles="cartodbpositron",  # Light, clean basemap style
    zoom_start=13
)

folium.GeoJson(
    filtered_data,
    name="geojson",
    tooltip=folium.GeoJsonTooltip(fields=['id'], sticky=False)
).add_to(m)
 
m

## Redo archetype naming convention to match HAZUS 6.0 inventory technical manual

In [ ]:
df

In [126]:
def assign_area_based_occupancies(row):
    
    area = row['PlanArea'] # ft2    
    
    # General Occupancy class (based on amount of sq feet), table 5-3
    Occupancy_ranges = {
        (0,2200): '1', # Single family
        (2200,4400):'2', # Duplex type of building
        (4400,8000):'3', # 3 to 4 units
        (8000,15000):'4', # 5 to 9 units
        (15000,40000):'5', # 10 to 19 units
        (40000,80000):'6', # 20 to 49 units
        (80000,5E10):'6', # 50 + units
    }    
    
    # Derive occupancy & house type :
    for bounds in Occupancy_ranges.keys(): 
        if bounds[0] <= area< bounds[1] : 
            Occupancy = Occupancy_ranges[bounds]    
    return(pd.Series(Occupancy))

def assign_hurricane_building_type(row) :
    
    stry_cnt = row['NumberOfStories']  # Number of storeys
    if int(stry_cnt) > 2 : 
        stry_cnt = 'X'
    occup = row['occupancies']         # Occupancy based on number on building size
    # Find the structure type 
    HAZUS_structural_types = {
        'WOD':'W',   # Wood
        'RCC':'C',   # Reinforced concrete
        'STL':'S',   # Steel
        'MAS':'M',   # Masonry
        #'MAB':'-',  # Manufactured homes are not incorporated yet.
    }
    Structural_type = HAZUS_structural_types[row['StructureType']]
    House_type = f'{Structural_type}-{occup}-{stry_cnt}'
    
    # Assemble relevant building ID
    building_IDs = {
        'W-1-1' : 'W.SF.1', # Wood, single family, one-story
        'W-1-2' : 'W.SF.2', # Wood, single family, two stories
        'W-1-X' : 'W.SF.2', # Wood, single family, more than two stories
        'W-2-1' : 'W.MUH.1', # Wood, multi-unit housing, one story  
        'W-2-2' : 'W.MUH.2', # Wood, multi-unit housing, two stories
        'W-2-X' : 'W.MUH.3', # Wood, multi-unit housing, three stories or more
        'M-1-1' : 'M.SF.1', # Masonry, single family, one-story
        'C-1-1' : 'C.ERB.L', # Concrete residential building, low-rise (1-2 stories)
        'C-2-1' : 'C.ERB.L', # Concrete residential building, low-rise (1-2 stories)
    }
    build_type = building_IDs[House_type]
    return(pd.Series(build_type))

def assign_roof_geometry(row) :
    shape = row['roofshape']
    
    Roof_shapes = {
        'Gable':'gab',
        'Hip':'hip',
        'Flat':'flt',    
    }     
    
    House_type = row['base_type']
    
    if (House_type in ['W.SF.1','W.SF.2']) and (shape == 'Flat') : 
        shape = 'Gable'  ## These house types do not have fragility curves. A proxy of Gable roof is returned.
    
    if row['base_type'] not in ['C.ERB.L',] : 
        House_type = f'{House_type}.{Roof_shapes[shape]}'
    
    return House_type

def assign_remaining_inputs(row) : 
    
    House_type = row['base_type']
    Options = {}
    
    if House_type[0:4] in ['W.SF','W.MU', 'M.SF'] : 
        # Wood framed houses have the following characteristics : (HAZUS technical manual 4.2.3)
        # Currently known characteristics : Wall construction ; No. of stories ; Roof shape
        
        # Unknowns : 
        # Secondary water protection (Yes/No) -- typically tar mopped roofing or bituminous strips along edges of roof sheating panels
        # Roof/wall connection (tnail/strap) -- toe nails or straps
        # Sheath nails (6d/6s/8d/8s) -- 6 and 8 refers to nail size, s = @6x6, d = @6x12 for spacing
        # Garage door (None/Strong/Weak)  -- unavailable feature for W.MUH buildings.
        # Shutters (Yes/No)
        # Masonry reinforcing (Yes/No)  -- available only for Masonry buildings.
        # Terrain roughness (3/15/35/70/100)
        #        3 : Open field (0.03 m)
        #       15 : Low density - shrubs and bushes (0.15 m)
        #       35 : Medium density urban (0.35 m)
        #       70 : High density and high-rise urban (0.70 m)
        #      100 : Dense Forests (1.00 m)
        
        #----------------------------------------------------        
        # Initiate variables : 
        SWP_key = 'No'
        Rf2WallRNG = random.random()
        Rf_cvr = 'None'
        Rf_nailsRNG = str(random.randint(2,3))
        Shutters_key = 'False'
        Garage = ''
        Terrain_rough_key = 0.35
        
        # W.MUH specific : 
        Rf_cover = ''
        Rf_condition = ''
        
        # M.SF specific : 
        MR = ''
        
        #----------------------------------------------------
        Secondary_water = {
            'Yes' : 1,
            'No'  : 0,
        }
        
        #SWP = str(Secondary_water[SWP_key])+'-opt'  ## Currently not conventional practice. 
        SWP = 'SWP-opt'
        
        
        Options[SWP] = ['1','0']
        #----------------------------------------------------           
        Roof_wall_connections = {
            'improved'  : 'strap', # Possible, not necessarily enforced by Canadian building codes
            'basic'  : 'tnail', # Default value            
        }
        
        if Rf2WallRNG < 0.9 : 
            Rf2Wall = 'basic'
        else :
            Rf2Wall = 'improved'

        # Derive the connection type (currently unknown) : ##### Guessed #####
        R2W = 'R2W-opt'
        
        Options[R2W] = ['strap','tnail'] 
        #----------------------------------------------------        
        # Deck nailing pattern :
        Roof_deck = {
            '1':'8s',    # 8d nails @ 6"/6"
            '2':'8d',    # 8d nails @ 6"/12" ##### Common nailing pattern according to Canadian Wood Council handbook/ CMHC  
            '3':'6d',    # 6d nails @ 6"/12" ##### Common nailing pattern according to Canadian Wood Council handbook/ CMHC
            '4':'6s',    # 6d nails @ 6"/6"
        }        
        
        # Derive the nailing pattern (currently unknown pattern) : ##### Guessed #####
        nails_pattern = 'Nail_pattern-opt'
        
        Options[nails_pattern] = ['8s','8d','6d','6s']
        #----------------------------------------------------         
        if House_type[0:4] not in ['W.MU',] : 
            Garage_availability = {
                1:'std',  # Standard garage (30 years of lifespan)
            #   'True':'wkd',  # Weakened garage (over 30 years of lifespan & not compliant to wind borne debris strenght)   
            #   'True':'sup',  # Superior garage resistance (under 30 years of lifespan & with a strenght requirement for debris)
                0:'no',
            }
            
            Garage = Garage_availability[row['garageExists']]
            if (House_type[0:4] in ['M.SF',]) & (Garage == 'no') : 
                Garage = 'nav'
        #------------------------------------------------------------------
        Shutters_availability ={
            'True' : '1',
            'False': '0',
        } 
       
        # Assumes no shutters (rare to non-existent for most buildings)
        Shutters = Shutters_availability[Shutters_key]
        #------------------------------------------------------------------    
        Terrain_roughness_available = {
            0.15 : '15',
            0.3  : '30',
            0.35 : '35',
            0.7  : '70',
            1  : '100'}    
        Terrain = Terrain_roughness_available[Terrain_rough_key]
        
        if House_type[0:4] in ['W.MU',] : 
            # Roof shape : 
            shape = row['building_type_with_roof'][-3:]
            
            if shape == 'flt' : 
                Roof_covers = {
                    'BUR':'bur',# Built-up roof (BUR) - multiple plies of roofing felts
                    'SPM' :'spm',# Single-ply membrane (SPM) - thermoplastic membranes, thermosets, modified bitumen membranes or liquid applied
                }             
                # State of the water protection : 
                Roof_condition = {
                    'Poor' : 'por',
                    'Good':'god',

                }

                Rf_cover ='Rf_cover-opt'
                Rf_condition = 'Rf_condition-opt'
                Options.pop(SWP) # Remove the secondary water option (not relevant for W.MUH buildings)
                
                SWP = 'null'
                Options[Rf_cover] = ['bur','spm']
                Options[Rf_condition] = ['por','god']
                
            else : 
                Rf_cover = 'null'
                Rf_condition = 'null'

        if House_type[0:4] in ['M.SF',] :
            MR = '0'  ## No masonry reinforcement
                
        
        archetype = row['building_type_with_roof']
        generic_archetype = row['building_type_with_roof']
        for feature in [Rf_cover,Rf_condition,SWP,nails_pattern,R2W,Garage,Shutters,MR,Terrain] : 
            if feature != '' : 
                archetype = archetype +'.'+str(feature)
                if feature not in Options : 
                    generic_archetype = generic_archetype+'.'+str(feature)
                else : 
                    generic_archetype = generic_archetype+'.'+'x'
    
    if House_type[0:5] in ['C.ERB',] :
        Rf_cover = ''
        Wndow_ratio = (random.random())*100
        Shutters_key = 'False'
        windborn_environment = 'residential' # Same environment for all buildings in the case study
        Terrain_rough_key = 0.35        
        #------------------------------------------------------------------         
        Roof_covers = {
            'BUR':'bur',# Built-up roof (BUR) - multiple plies of roofing felts
            'SPM' :'spm',# Single-ply membrane (SPM) - thermoplastic membranes, thermosets, modified bitumen membranes or liquid applied
        }
        

        Rf_cover = 'Rf_cover-opt'
        
        Options[Rf_cover] = ['bur','spm']
        #------------------------------------------------------------------ 
        Window_area_ratio = {
            '0-33': 'low',
            '33-50': 'med',           
            '50-100': 'hig',          
        }
        if 0 <Wndow_ratio < 33 : 
            Wndow_key = '0-33'
        elif 33 < Wndow_ratio < 50 : 
            Wndow_key = '33-50'
        elif 50 < Wndow_ratio :
            Wndow_key = '50-100'
            
        Wndow = 'Wndow-opt'
        
        Options[Wndow] = ['low','med','hig']
        #------------------------------------------------------------------
        Shutters_availability ={
            'True' : '1',
            'False': '0',
        } 
       
        # Assumes no shutters (rare to non-existent for most buildings)
        Shutters = Shutters_availability[Shutters_key]        
        
        #------------------------------------------------------------------ 
        # Missile environment : 
        # A : Mixed residential-commercial windborn debris missile environment
        # B : windborn debris varies by direction
        # C : residential windborn debris
        # D : No windborn debris
        Windborn_environments = {
            'mixed-res_com':'A',
            'directional':'B',
            'residential':'C',
            'None':'D',
        }
        debris = Windborn_environments[windborn_environment]
        #------------------------------------------------------------------    
        Terrain_roughness_available = {
            0.15 : '15',
            0.3  : '30',
            0.35 : '35',
            0.7  : '70',
            1  : '100'}    
        Terrain = Terrain_roughness_available[Terrain_rough_key]
        
        archetype = House_type
        generic_archetype = House_type
        for feature in [Rf_cover,Wndow,Shutters,debris,Terrain] : 
            if feature != '' : 
                archetype = archetype +'.'+str(feature)
                if feature not in Options : 
                    generic_archetype = generic_archetype+'.'+str(feature)
                else : 
                    generic_archetype = generic_archetype+'.'+'x'
    

    keys,values = zip(*Options.items())
    permutations_dicts = [dict(zip(keys,v)) for v in it.product(*values)]
    
    potential_archetypes = []
      
    for combo in permutations_dicts :
        modified_archetype = copy.copy(archetype)        
        for k,v in combo.items() : 
            modified_archetype = modified_archetype.replace(k,v)
        potential_archetypes.append(modified_archetype)    

    guessed_archetype = archetype.replace('-opt','')

    return pd.Series([potential_archetypes, guessed_archetype,generic_archetype])






In [127]:
df['occupancies'] = df.apply(assign_area_based_occupancies,axis =1)

In [128]:
df['base_type']= df.apply(assign_hurricane_building_type,axis =1)

In [129]:
df['building_type_with_roof'] = df.apply(assign_roof_geometry,axis =1)

In [ ]:
df[['Potential archetypes','Guessed archetype','Generic archetype']]=df.apply(assign_remaining_inputs,axis =1)
display(df)

## Add LCA specific information on exterior building material

In [ ]:
# Manual assessment of the building claddings
building_claddings = pd.read_excel('BRAILS - facades.xlsx',index_col = 0)
# 1 building with paneling cladding converted to vinyl
# 3-4 buildings with plaster cladding converted to brick

# Merge with current dataframe :
df = pd.concat([df,building_claddings],axis = 1)
display(df)

In [132]:
building_claddings['cladding'].unique().tolist()

['Vinyl', 'Cedar', 'Brick']

## Output the results

In [133]:
df.to_excel('Archetypes large building inv.xlsx', index= True)